In [ ]:
# Tåg och trafikanalys Stockholm

Importering av bibliotek

Här importerar jag de bibliotek som jag kommer ha användning för detta projekt vilket inkluderar os, csv, requests, matplotlib och datetime.

In [ ]:
# ==============================================================================================================
# Importering av bibliotek
# ==============================================================================================================

import os
import csv
import requests
import matplotlib.pyplot as plt

from datetime import datetime   # Hämtar dagens datum
from datetime import timedelta  # Hämtar morgondagens datum
from getpass import getpass # Hämtar API-nyckeln från användaren om den inte finns i miljövariablerna


print("Alla bibliotek har importerats!")

API-Nyckeln

In [ ]:
# ==============================================================================================================
# API-nyckel
# ==============================================================================================================

API_KEY = "ad1dd6ea9368474cb498c74084e8573e"

print("API-nyckeln har lästs in!")


Importering av API

In [ ]:
def get_station_locations(station_name):
    """
    Söker efter stationer med ResRobot API.
    """

    url = "https://api.resrobot.se/v2.1/location.name"

    params = {
        "input": station_name,
        "maxNo": 10,
        "lang": "sv",
        "format": "json",
        "accessId": API_KEY
    }

    try:

        response = requests.get(
            url,
            params=params,
            timeout=10
        )

        response.raise_for_status()

        data = response.json()

        return data.get("StopLocation", [])

    except requests.exceptions.RequestException as e:

        print(f"Ett API-fel uppstod: {e}")

        return []


def get_departures(stop_id, date_str):
    """
    Hämtar avgångar från vald station.
    """

    url = "https://api.resrobot.se/v2.1/departureBoard"

    # För dagens datum börjar vi ungefär från aktuell tid.
    # För framtida datum börjar vi vid midnatt.

    today = datetime.now().strftime("%Y-%m-%d")

    if date_str == today:
        time_str = datetime.now().strftime("%H:%M")
    else:
        time_str = "00:00"

    params = {
        "id": stop_id,
        "format": "json",
        "accessId": API_KEY,
        "date": date_str,
        "time": time_str,
        "duration": 120,
        "maxJourneys": 40,
        "lang": "sv"
    }

    try:

        response = requests.get(
            url,
            params=params,
            timeout=10
        )

        response.raise_for_status()

        return response.json()

    except requests.exceptions.RequestException as e:

        print(f"Ett API-fel uppstod: {e}")

        return {}


print("Importeringen av API:n är klar!")


Föräldrar klass

In [ ]:
# ==============================================================================================================
# Föräldrarklass
# ==============================================================================================================

class Departure:
    """
    Föräldraklass som representerar en avgång.
    """

    def __init__(self, station, destination, date, scheduled_time, realtime_time, realtime_date, line, transport_category, operator):
        
        self.station = station
        self.destination = destination
        self.date = date
        self.scheduled_time = scheduled_time
        self.realtime_time = realtime_time
        self.realtime_date = realtime_date
        self.line = line
        self.transport_category = transport_category
        self.operator = operator


    def get_delay(self):
        """
        Räknar ut försening i minuter.
        """

        # Om realtidsdata saknas
        if not self.realtime_time:
            return None

        # Om datum saknas
        if not self.date:
            return None

        # Om realtidsdatum saknas används planerat datum
        realtime_date = self.realtime_date

        if not realtime_date:
            realtime_date = self.date

        try:

            scheduled_datetime = datetime.strptime(
                f"{self.date} {self.scheduled_time}",
                "%Y-%m-%d %H:%M:%S"
            )

            realtime_datetime = datetime.strptime(
                f"{realtime_date} {self.realtime_time}",
                "%Y-%m-%d %H:%M:%S"
            )

            difference = (
                realtime_datetime - scheduled_datetime
            ).total_seconds() / 60

            # Negativ försening räknas som 0
            return max(0, int(round(difference)))

        except (ValueError, TypeError):

            return None


    def is_delayed(self):
        """
        Kontrollerar om tåget är försenat.
        """

        delay = self.get_delay()

        if delay is None:
            return False

        return delay > 0


# =================================================================================================================

class TrafficAnalyzer:
    """
    Föräldraklass som analyserar avgångarna.
    """

    def __init__(self, departure_list):

        self.departure_list = departure_list


    def get_realtime_departures(self):
        """
        Returnerar bara avgångar som har realtidsinformation.
        """

        return [
            departure
            for departure in self.departure_list
            if departure.get_delay() is not None
        ]


    def find_most_delayed(self):
        """
        Hittar det mest försenade tåget.
        """

        departures = self.get_realtime_departures()

        if not departures:
            return None

        return max(
            departures,
            key=lambda x: x.get_delay()
        )


    def find_least_delayed(self):
        """
        Hittar det tåg som är minst försenat.
        """

        departures = self.get_realtime_departures()

        if not departures:
            return None

        return min(
            departures,
            key=lambda x: x.get_delay()
        )


    def average_delay(self):
        """
        Räknar ut genomsnittlig försening.
        """

        departures = self.get_realtime_departures()

        if not departures:
            return 0

        total = sum(
            x.get_delay()
            for x in departures
        )

        return total / len(departures)


    def calculate_difference(self):
        """
        Räknar ut skillnaden mellan
        största och minsta försening.
        """

        most_delayed = self.find_most_delayed()
        least_delayed = self.find_least_delayed()

        if not most_delayed or not least_delayed:
            return 0

        return (
            most_delayed.get_delay()
            - least_delayed.get_delay()
        )


    def count_delayed(self):
        """
        Räknar antalet försenade avgångar.
        """

        departures = self.get_realtime_departures()

        return sum(
            1
            for departure in departures
            if departure.get_delay() > 0
        )


    def count_on_time(self):
        """
        Räknar antalet avgångar som är i tid.
        """

        departures = self.get_realtime_departures()

        return sum(
            1
            for departure in departures
            if departure.get_delay() == 0
        )




Barnklass

In [ ]:
# ==============================================================================================================
# Barnklass
# ==============================================================================================================

# Barnklass som ärver från Departure och returnerar status beroende på om tåget är i tid eller försenat
class TrainDeparture(Departure):

    def get_status_text(self):
        delay = self.get_delay()

        if delay is None:
            return "Ingen realtidsinformation"

        elif delay == 0:
            return "I tid"

        elif delay <= 5:
            return "Mindre försening"

        else:
            return "Försenad"

Funktioner

In [ ]:
# ==============================================================================================================
# Funktioner
# ==============================================================================================================

#  Ber användaren skriva en station och väljer en station från API-resultatet.
def validate_station():

    while True:

        station_name = input("\nAnge station, exempelvis Stockholm: ")
        station_name = station_name.strip()

        if not station_name:
            print("Du måste ange en station!")
            continue

        results = get_station_locations(station_name)

        if not results:
            print("Kunde inte hitta stationen.")
            print("Försök igen!")
            continue

        print("\nHittade stationer:")
        print("-" * 50)

        for index, station in enumerate(results, start=1):
            print(f"{index}. " f"{station.get('name')} " f"(ID: {station.get('extId')})")


        while True:
            choice = input(f"\nVälj station mellan 1-{len(results)}: ")

            if (choice.isdigit() and 1 <= int(choice) <= len(results)):
                return results[int(choice) - 1]

            else:
                print("Ogiltigt val, försök igen!")

# =================================================================================================================

# Skapar TrainDeparture-objekt från API-datan
def create_departure_objects(raw_data, station_name):
    objects = []    # Skapar en tom lista för att lagra TrainDeparture-objekten

    departures = raw_data.get("Departure", [])

    for item in departures:

        # Hämtar Product-information
        product = item.get("Product", {})

        # ResRobot använder Product som objekt.
        # Om API:t returnerar en lista tar vi första objektet.
        if isinstance(product, list):

            if product:
                product = product[0]

            else:
                product = {}

        # Hämtar typ av trafik
        transport_category = product.get("catOutL", "")


        # Vi analyserar endast tåg.
        if "tåg" not in transport_category.lower():
            continue

        # Planerad tid
        scheduled_time = item.get("time", "")

        # Datum
        date = item.get("date", "")

        # Realtid
        realtime_time = item.get("rtTime", None)

        # Realtidsdatum
        realtime_date = item.get("rtDate", date)

        # Linjenummer
        line = item.get("transportNumber", "Okänd")

        # Destination
        destination = item.get("direction", "Okänd destination")

        # Operatör
        operator = product.get("operator", "Okänd")


        # Skapar TrainDeparture-objekt
        obj = TrainDeparture(
            station=station_name,
            destination=destination,
            date=date,
            scheduled_time=scheduled_time,
            realtime_time=realtime_time,
            realtime_date=realtime_date,
            line=line,
            transport_category=transport_category,
            operator=operator
        )

        objects.append(obj)

    return objects


# =================================================================================================================

# Hämtar, analyserar och sparar trafikdata
def analyze_traffic(station, date_str):
    station_name = station.get("name", "Okänd station")

    stop_id = station.get("extId")

    print(f"\nHämtar tågavgångar från ", f"{station_name}...")
    print(f"Datum: {date_str}\n")

    # Hämtar API-data
    raw_data = get_departures(stop_id, date_str)

    if not raw_data:
        print("Kunde inte hitta någon trafikdata :(")
        return

    # Skapar objekt
    objects = create_departure_objects(raw_data, station_name)

    if not objects:
        print("Kunde inte hitta några tågavgångar.")
        print("Stationen kanske inte har tågtrafik ", "eller så saknas data.")
        return

    # Skapar analyserare
    analyzer = TrafficAnalyzer(objects)

    # Hämtar analysresultat
    most_delayed = (analyzer.find_most_delayed())

    least_delayed = (analyzer.find_least_delayed())

    average = (analyzer.average_delay())

    difference = (analyzer.calculate_difference())

    print("\n" + "-" * 50)
    print(f"Analys för {station_name}")
    print(f"Datum: {date_str}")
    print("-" * 50)


    # Antal avgångar
    print(f"Antal tågavgångar: ", f"{len(objects)}")

    # Medelförsening
    if average > 0:
        print(f"Genomsnittlig försening: ", f"{average:.1f} minuter")

    else:
        print("Genomsnittlig försening: ", "0.0 minuter")

    # Antal försenade
    print(f"Antal försenade avgångar: ", f"{analyzer.count_delayed()}")

    # Antal i tid
    print(f"Antal avgångar i tid: ", f"{analyzer.count_on_time()}")

    # Mest försenade
    if most_delayed:
        print("\nMest försenade tåg:")
        print(f"Linje: ", f"{most_delayed.line}")
        print(f"Destination: ", f"{most_delayed.destination}")
        print(f"Operatör: ", f"{most_delayed.operator}")
        print(f"Försening: ", f"{most_delayed.get_delay()} minuter")
        print(f"Status: ", f"{most_delayed.get_status_text()}")

    # Minst försenade
    if least_delayed:
        print("\nMinst försenade tåg:")
        print(f"Linje: ", f"{least_delayed.line}")
        print(f"Destination: ", f"{least_delayed.destination}")
        print(f"Försening: ", f"{least_delayed.get_delay()} minuter")

    # Skillnad
    print(f"\nSkillnad mellan största ", f"och minsta försening: ", f"{difference} minuter")

    # Spara CSV
    save_to_csv(objects)

    # Visa diagram
    print_diagram(objects, station_name, date_str)


# =================================================================================================================

# Skapar diagram över tågens förseningar
def print_diagram(departure_list, station_name, date_str):
    times = []      # Skapar en tom lista för att lagra x-axelns etiketter
    delays = []     # Skapar en tom lista för att lagra förseningar

    for departure in departure_list:
        delay = departure.get_delay()

        # Hoppa över avgångar som saknar realtidsdata
        if delay is None:
            continue

        # Skapar text för x-axeln
        times.append(f"{departure.line}\n", f"{departure.scheduled_time[:5]}")

        delays.append(delay)

    # Om det saknas diagramdata
    if not delays:
        print("\nDet finns ingen realtidsdata ", "att visa i diagrammet.")
        return

# Skpandet av diagrammet
    plt.figure(figsize=(12, 6))
    plt.bar(range(len(delays)), delays)

    # Rubrik
    plt.title(f"Tågförseningar vid ", f"{station_name} ", f"{date_str}")

    # Axeltexter
    plt.xlabel("Tågavgång")
    plt.ylabel("Försening i minuter")

    # X-axel
    plt.xticks(range(len(times)), times, rotation=45)
    
    plt.tight_layout()
    plt.show()

Funktion som sparan till en CSV fil

In [ ]:
# ==============================================================================================================
# Spara till CSV
# ==============================================================================================================

# Sparar tågdata i en CSV-fil
def save_to_csv(departure_list, filename="trafikdata.csv"):
    try:
        file_exists = os.path.exists(filename)

        with open(filename, "a", newline="", encoding="utf-8") as file:    # 'a' används för att gamla data ska finnas kvar
            writer = csv.writer(file)

            # Skriver rubriker om filen är ny
            if not file_exists:
                writer.writerow([
                    "station",
                    "date",
                    "scheduled_time",
                    "realtime_time",
                    "line",
                    "destination",
                    "operator",
                    "transport_category",
                    "delay"
                ])


            # Skriver varje avgång
            for departure in departure_list:
                delay = departure.get_delay()

                writer.writerow([
                    departure.station,
                    departure.date,
                    departure.scheduled_time,
                    departure.realtime_time
                    
                    if departure.realtime_time
        
                    else "",
                    departure.line,
                    departure.destination,
                    departure.operator,
                    departure.transport_category,
                    delay
                    if delay is not None
                    else ""
                ])


        print(f"\nData har sparats till {filename}")


    except OSError as e:
        print(f"Kunde inte spara CSV-filen: {e}")


Menyn

Här kan användaren välja vad progrmamet ska göra. Antingen analysera dagens avgångar (alternativ 1) eller analysera morgondagens avgångar (alternativ 2).

In [ ]:
# ==============================================================================================================
# Menyval
# ==============================================================================================================

while True:

    print("\n" + "-" * 35)
    print("Välkommen till Tåg- och trafikanalysen!")
    print("-" * 35)

    print("1. Analysera dagens tågavgångar")
    print("2. Analysera morgondagens tågavgångar")
    print("3. Avsluta programmet")

    val = input("Välj ett alternativ mellan 1-3: ")


    if val == "1":
        # Hämtar dagens datum
        date = datetime.now().strftime("%Y-%m-%d") 

        station = validate_station()

        print(f"\nHämtar tågavgångar för {date} i {station['name']}...\n")

        analyze_traffic(station, date)

# ==============================================================================================================

    elif val == "2":
        # Hämta morgondagens datum
        date_tomorrow = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d") 

        station = validate_station()

        print(f"\nHämtar tågavgångar för {date_tomorrow} i {station['name']}...\n")

        analyze_traffic(station, date_tomorrow)

# ==============================================================================================================
        
    elif val == "3":

        print("\nProgrammet avslutas...")

        break

# ==============================================================================================================

    else:

        print("\nOgiltigt val, vänligen försök igen!\n")

Du måste ange en station!
Du måste ange en station!
Du måste ange en station!
Du måste ange en station!
Du måste ange en station!
Du måste ange en station!
Du måste ange en station!
Du måste ange en station!
Du måste ange en station!
Du måste ange en station!
Du måste ange en station!
Du måste ange en station!
Du måste ange en station!
Du måste ange en station!
Du måste ange en station!
Du måste ange en station!
Du måste ange en station!
Du måste ange en station!
Ett API-fel uppstod: 401 Client Error: Unauthorized for url: https://api.resrobot.se/v2.1/location.name?input=Stockholm&maxNo=10&lang=sv&format=json&accessId=ad1dd6ea9368474cb498c74084e8573e
Kunde inte hitta stationen.
Försök igen!
Du måste ange en station!
